# NV5 — Dự báo chiếm dụng cổng và số phiên bắt đầu

Thực nghiệm hồi quy chuỗi thời gian 15 phút, dự báo một bước. Đây là dự báo từ lịch và trạng thái quan sát đến khung trước, không phải dự báo cả ngày tại một thời điểm.

Mapping vị trí theo từng phiên dùng `station_name,cluster_id`; mã cổng vật lý chuẩn hóa riêng. Các mốc chia nằm trong `reports/nv5_splits.json`. Chạy lần lượt; cell audit không huấn luyện, cell cuối mới fit toàn bộ mô hình.


## 1. Thiết lập và thư viện

In [1]:
"""NV5: 15-minute station-cluster port occupancy and session-arrival forecasts.

Run each notebook cell in order after the team supplies the approved station-cluster
mapping and split milestones.

The mapping file must contain station_name,cluster_id. The split JSON must contain
train_end_utc, validation_end_utc, and test_end_utc (exclusive boundaries).
"""
from __future__ import annotations

import argparse
import time
import hashlib
import platform
from importlib.metadata import version
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while not (ROOT / 'src' / 'config.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Open this notebook from the project workspace')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import (RAW_DATA_FILE, SESSIONS_CLEAN_PARQUET,
                        SESSIONS_FEATURES_PARQUET)

OUT = ROOT / "outputs" / "nv5"
STEP = pd.Timedelta(minutes=15)
SEED = 42


def _as_datetime(series):
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


def _expected_port_id(df):
    mac = df["MAC Address"].astype("string").str.strip().str.upper().str.replace(r"[^0-9A-F]", "", regex=True)
    port = df["Port Number"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    good = mac.notna() & port.notna() & mac.ne("") & port.ne("")
    result = pd.Series(pd.NA, index=df.index, dtype="string")
    result.loc[good] = mac.loc[good] + "_Port_" + port.loc[good]
    return result

def _normalize_port_id(values):
    parts = values.astype("string").str.rsplit("_Port_", n=1, expand=True)
    if parts.shape[1] != 2:
        return pd.Series(pd.NA, index=values.index, dtype="string")
    return _expected_port_id(pd.DataFrame({"MAC Address": parts[0], "Port Number": parts[1]}))


def clip_to_port_limit(values, port_limit=None):
    # Compatibility helper. Evaluation never uses a future-derived capacity cap.
    return np.clip(np.asarray(values, dtype=float), 0, port_limit)


## 2. Đầu vào, cổng vật lý và cụm vị trí

Áp dụng mục 3.5.1: MAC viết hoa, bỏ dấu phân cách; số cổng bỏ hậu tố `.0`. Giữ nguyên parquet NV1. Mã cổng lưu sẵn phải khớp sau chuẩn hóa. `cluster_id` lấy theo Station Name của từng phiên từ bảng 47 tên trạm → 9 cụm, không lấy nhãn cụm hành vi NV2.

Cổng chuyển từ HAMILTON sang BRYANT giữ cùng định danh vật lý nhưng đổi cụm theo phiên. Phiên hợp lệ còn thiếu định danh làm dừng bước gán cụm.

**Nghĩa của mục tiêu:** occupancy là số cổng riêng biệt có phiên giao với khung, không phải phép đo đồng thời tại một thời điểm hay tỷ lệ sử dụng công suất thiết kế. Khoảng kết nối dùng `[start,end)`; mỗi cổng chỉ đếm một lần mỗi khung.

Không có lịch sử khả dụng thiết bị. Điền 0 nghĩa là không quan sát thấy phiên trong dữ liệu; không chứng minh trạm hoạt động và còn cổng trống. Không chặn dự báo theo tổng số cổng quan sát toàn lịch sử vì đây không phải sức chứa đã biết theo thời điểm.


In [2]:
def load_sessions():
    """Prefer NV1 features, compare its row/valid counts with clean parquet."""
    parquet_rejection = None
    if SESSIONS_FEATURES_PARQUET.exists():
        try:
            features = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
            clean = (pd.read_parquet(SESSIONS_CLEAN_PARQUET)
                     if SESSIONS_CLEAN_PARQUET.exists() else None)
            if clean is not None:
                k = ["MAC Address", "Port Number", "Start Date", "End Date"]
                audit = {
                    "clean_rows": len(clean), "features_rows": len(features),
                    "clean_valid_sessions": int(clean["is_valid_session"].sum()),
                    "features_valid_sessions": int(features["is_valid_session"].sum()),
                    "same_session_keys": bool(clean[k].sort_values(k).reset_index(drop=True).equals(
                        features[k].sort_values(k).reset_index(drop=True))),
                }
                if (audit["clean_rows"] != audit["features_rows"] or
                    audit["clean_valid_sessions"] != audit["features_valid_sessions"] or
                    not audit["same_session_keys"]):
                    raise ValueError(f"NV1 clean/features mismatch: {audit}")
            else:
                audit = {"clean_parquet_available": False}
            print("Source: NV1 sessions_features.parquet; audit:", audit)
            return features, "nv1_parquet", audit
        except (ImportError, ValueError, OSError) as exc:
            parquet_rejection = str(exc)
            print("NV1 parquet unavailable/inconsistent:", str(exc)[:300])

    from src.feature_engineering import build_clean_pipeline
    print("Fallback: rebuild NV1 cleaning flags from raw CSV")
    raw = pd.read_csv(RAW_DATA_FILE, low_memory=False)
    data, initial, duplicates = build_clean_pipeline(raw)
    audit = {"raw_rows": initial, "exact_duplicates_removed": duplicates,
             "clean_rows": len(data),
             "valid_sessions": int(data["is_valid_session"].sum()),
             "parquet_rejection_reason": parquet_rejection}
    return data, "nv1_raw_recomputed", audit


def attach_clusters(df, mapping_path):
    """Apply report 3.5.1: physical port normalization; location per session."""
    required = {"MAC Address", "Port Number", "Station Name"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing NV1 columns: {sorted(required - set(df.columns))}")
    df = df.copy()
    expected = _expected_port_id(df)
    if "port_id" in df:
        actual = _normalize_port_id(df["port_id"])
        mismatch = expected.notna() & df["port_id"].notna() & actual.ne(expected).fillna(True)
        if mismatch.any():
            raise ValueError("Stored port_id disagrees with MAC Address + Port Number")
    df["port_id"] = expected
    mapping_path = Path(mapping_path)
    if not mapping_path.exists():
        mapping_path.parent.mkdir(parents=True, exist_ok=True)
        pd.read_csv(ROOT / "reports/station_cluster_map_example.csv", dtype="string").to_csv(mapping_path, index=False)
    mapping = pd.read_csv(mapping_path, dtype="string")
    required_map = ["station_name", "cluster_id"]
    if not set(required_map).issubset(mapping.columns):
        raise ValueError("Cluster mapping requires station_name,cluster_id (location per session)")
    mapping = mapping[required_map].apply(lambda col: col.str.strip())
    if mapping.isna().any().any() or mapping.eq("").any().any():
        raise ValueError("Cluster mapping contains missing or blank station/cluster IDs")
    if mapping.station_name.duplicated().any():
        raise ValueError("Each station_name must map to exactly one cluster_id")
    station = df["Station Name"].astype("string").str.strip()
    df["cluster_id"] = station.map(mapping.set_index("station_name").cluster_id)
    valid = df.get("is_valid_session", pd.Series(True, index=df.index)).fillna(False)
    missing = valid & (df.port_id.isna() | df.cluster_id.isna())
    if missing.any():
        raise ValueError(f"{int(missing.sum())} valid sessions lack port/cluster IDs; stations: {station[missing].drop_duplicates().tolist()}")
    df.attrs["cluster_mapping"] = {
        "mapping_key": "station_name", "mapped_station_names": len(mapping),
        "mapped_ports": int(df.port_id.nunique()),
        "clusters": sorted(mapping.cluster_id.unique().tolist()),
        "ports_in_multiple_locations": int(df.groupby("port_id").cluster_id.nunique().gt(1).sum()),
        "valid_sessions_without_ids": int(missing.sum()),
    }
    return df


def make_regular_series(df):
    """Create cluster-local counts on a regular UTC grid using [start,end)."""
    required = ["start_datetime_utc", "end_datetime_utc", "is_valid_session",
                "port_id", "cluster_id"]
    absent = [c for c in required if c not in df]
    if absent:
        raise ValueError(f"Missing required columns: {absent}")
    valid_source = df.loc[df.is_valid_session.fillna(False)].copy()
    missing_port_count = int(valid_source.port_id.isna().sum())
    missing_cluster_count = int(valid_source.cluster_id.isna().sum())
    data = valid_source.copy()
    data["start_datetime_utc"] = _as_datetime(data.start_datetime_utc)
    data["end_datetime_utc"] = _as_datetime(data.end_datetime_utc)
    nonpositive_intervals = int((data.end_datetime_utc <= data.start_datetime_utc).fillna(False).sum())
    data = data.dropna(subset=["start_datetime_utc", "end_datetime_utc", "cluster_id"])
    data = data.loc[data.end_datetime_utc > data.start_datetime_utc]
    if data.empty:
        raise ValueError("No valid, mapped sessions remain")
    left = data.start_datetime_utc.min().floor("15min")
    right = data.end_datetime_utc.max().ceil("15min")
    index = pd.date_range(left, right - STEP, freq="15min")
    frames = []
    for cluster, group in data.groupby("cluster_id", sort=True):
        # Invalid/missing port IDs do not count toward physical occupancy.
        known = group.loc[group.port_id.notna() & group.port_id.astype("string").str.strip().ne("")]
        occupancy = np.zeros(len(index), dtype=np.int64)
        # Each port contributes at most 1 in a bin, even when sessions overlap.
        for _, port_sessions in known.groupby("port_id", sort=False):
            diffs = np.zeros(len(index) + 1, dtype=np.int64)
            starts = ((port_sessions.start_datetime_utc - left) // STEP).to_numpy(dtype=int)
            ends = ((port_sessions.end_datetime_utc - left - pd.Timedelta(1, "ns")) // STEP + 1).to_numpy(dtype=int)
            starts = np.clip(starts, 0, len(index))
            ends = np.clip(ends, 0, len(index))
            np.add.at(diffs, starts, 1)
            np.add.at(diffs, ends, -1)
            occupancy += np.cumsum(diffs[:-1]) > 0

        arrivals = np.zeros(len(index), dtype=np.int64)
        starts = ((group.start_datetime_utc - left) // STEP).to_numpy(dtype=int)
        in_range = (starts >= 0) & (starts < len(index))
        np.add.at(arrivals, starts[in_range], 1)
        frames.append(pd.DataFrame({"cluster_id": cluster, "timestamp_utc": index,
                                    "occupancy_ports": occupancy,
                                    "session_arrivals": arrivals,
                                    "observed_physical_ports": int(known.port_id.nunique())}))
    result = pd.concat(frames, ignore_index=True)
    info = {"valid_mapped_sessions": int(len(data)),
            "valid_sessions_excluded_nonpositive_interval": nonpositive_intervals,
            "valid_sessions_missing_port_id": missing_port_count,
            "valid_sessions_without_cluster_id": missing_cluster_count,
            "observed_physical_ports": int(data.port_id.nunique()),
            "first_utc": str(left), "last_utc_exclusive": str(right),
            "time_bins_per_cluster": int(len(index)),
            "clusters": sorted(data.cluster_id.astype(str).unique().tolist()),
            "max_occupancy": int(result.occupancy_ports.max()),
            "occupancy_exceeds_observed_ports": bool((result.occupancy_ports > result.observed_physical_ports).any())}
    if info["occupancy_exceeds_observed_ports"]:
        raise AssertionError("Cluster occupancy exceeds its observed physical port count")
    return result, info


## 3. Tạo chuỗi occupancy và arrivals theo cụm

In [3]:
def add_features(series, target):
    frames = []
    for cluster, group in series.groupby("cluster_id", sort=False):
        g = group.sort_values("timestamp_utc").set_index("timestamp_utc")
        s = g[[target]].copy()
        t = s.index
        if not t.is_unique or not (t.to_series().diff().dropna() == STEP).all():
            raise ValueError("Each cluster must have a unique regular 15-minute UTC grid")
        local = t.tz_localize("UTC").tz_convert("America/Los_Angeles")
        slot = local.hour * 4 + local.minute // 15
        dow = local.dayofweek
        s["slot"] = slot.astype(int)
        s["dow"] = dow.astype(int)
        s["month"] = local.month.astype(int)
        s["year"] = local.year.astype(int)
        s["post_policy"] = (local.tz_localize(None) >= pd.Timestamp("2017-08-01")).astype(int)
        s["weekend"] = (dow >= 5).astype(int)
        s["hour_sin"] = np.sin(2*np.pi*slot/96)
        s["hour_cos"] = np.cos(2*np.pi*slot/96)
        s["week_sin"] = np.sin(2*np.pi*dow/7)
        s["week_cos"] = np.cos(2*np.pi*dow/7)
        for lag in (1, 2, 4, 96, 672):
            s[f"lag_{lag}"] = s[target].shift(lag)
        for window in (4, 12, 96, 672):
            s[f"rolling_mean_{window}"] = s[target].shift(1).rolling(window, min_periods=window).mean()
        s = s.dropna().copy()
        s["cluster_id"] = cluster
        frames.append(s.reset_index(names="timestamp_utc"))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## 4. Kiểm soát thông tin tại thời điểm dự báo

Đối với khung t, lag và rolling chỉ dùng mục tiêu đến t−1. Rolling luôn `shift(1)` trước khi tính. Lịch sử được cập nhật bằng quan sát thật sau mỗi khung; mô hình không fit lại trên test/stress. Giao thức này giả định có quan sát kịp thời, cần kiểm chứng thêm nếu triển khai thực tế.

Lịch giờ/thứ theo America/Los_Angeles (có DST); lưới và mốc chia theo UTC. Chọn alpha Ridge và cấu hình XGBoost theo RMSE validation; test/stress không dùng chọn cấu hình. Đây là một lần chia theo thời gian, chưa phải kiểm chứng ổn định qua nhiều cửa sổ rolling-origin.

R² trên chuỗi hằng có thể bằng 1 theo quy ước thư viện, nên xuất riêng trung bình trên tất cả cụm và trên các cụm có mục tiêu không hằng. Chênh lệch nhỏ giữa mô hình chưa chứng minh khác biệt có ý nghĩa thống kê.


In [4]:
def metrics(y, p):
    y = np.asarray(y, dtype=float)
    return {"MAE": float(mean_absolute_error(y, p)),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": float(r2_score(y, p)),
            "constant_target": bool(np.ptp(y) == 0)}


def _parse_splits(splits):
    keys = ("train_end_utc", "validation_end_utc", "test_end_utc")
    train_end, validation_end, test_end = [pd.to_datetime(splits[k], utc=True).tz_convert(None) for k in keys]
    if not train_end < validation_end < test_end:
        raise ValueError("Require train_end_utc < validation_end_utc < test_end_utc")
    stress = splits.get("stress_start_utc")
    stress = pd.to_datetime(stress, utc=True).tz_convert(None) if stress is not None else None
    if stress is not None and stress < test_end:
        raise ValueError("stress_start_utc must be at or after test_end_utc")
    if splits.get("stress_end_utc") is not None:
        end = pd.to_datetime(splits["stress_end_utc"], utc=True).tz_convert(None)
        if stress is None or end <= stress:
            raise ValueError("stress_end_utc must follow stress_start_utc")
    return train_end, validation_end, test_end, stress


def ridge_predictions(train, validation, test, features, target, alpha=10.0):
    scaler = StandardScaler().fit(train[features])
    model = Ridge(alpha=alpha).fit(scaler.transform(train[features]), train[target])
    return model.predict(scaler.transform(validation[features])), model.predict(scaler.transform(test[features])), scaler


def run_target(series, target, splits, *, models=None):
    from xgboost import XGBRegressor
    train_end, validation_end, test_end, stress_start = _parse_splits(splits)
    data = add_features(series, target)
    cols = [c for c in data if c not in {target, "cluster_id", "timestamp_utc"}]
    records, frames, timings, candidates, importance = [], [], [], [], []
    for cluster, d in data.groupby("cluster_id", sort=True):
        d = d.sort_values("timestamp_utc")
        train = d.loc[d.timestamp_utc < train_end]
        parts = {"validation": d.loc[(d.timestamp_utc >= train_end) & (d.timestamp_utc < validation_end)],
                 "test": d.loc[(d.timestamp_utc >= validation_end) & (d.timestamp_utc < test_end)]}
        if stress_start is not None:
            mask = d.timestamp_utc >= stress_start
            if splits.get("stress_end_utc"):
                mask &= d.timestamp_utc < pd.to_datetime(splits["stress_end_utc"], utc=True).tz_convert(None)
            parts["stress_2020"] = d.loc[mask]
        if train.empty or any(part.empty for part in parts.values()):
            raise ValueError(f"Empty chronological split for {cluster}")
        assert train.timestamp_utc.max() < parts["validation"].timestamp_utc.min()
        assert parts["validation"].timestamp_utc.max() < parts["test"].timestamp_utc.min()
        predictions = {}
        if models is None or "Historical Mean" in models:
            hist = train.groupby(["dow", "slot"])[target].mean()
            default = float(train[target].mean())
            predictions["Historical Mean"] = {name: np.array([hist.get((int(r.dow), int(r.slot)), default)
                for r in part.itertuples()]) for name, part in parts.items()}
        for label, lag in (("Persistence t-1", 1), ("Persistence Day t-96", 96), ("Persistence Week t-672", 672)):
            if models is None or label in models:
                predictions[label] = {name: part[f"lag_{lag}"].to_numpy() for name, part in parts.items()}
        configs = {"Ridge": [{"alpha": a} for a in (0.1, 1, 10, 100)],
                   "Random Forest": [{"n_estimators": 60, "max_depth": 12, "min_samples_leaf": 4}],
                   "XGBoost": [{"n_estimators": 100, "max_depth": 3}, {"n_estimators": 160, "max_depth": 4}]}
        for name, grid in configs.items():
            if models is not None and name not in models:
                continue
            if name == "Ridge":
                scaler = StandardScaler().fit(train[cols])
                x_train = scaler.transform(train[cols])
                x_val = scaler.transform(parts["validation"][cols])
            best, best_error, best_params = None, float("inf"), None
            for params in grid:
                start = time.perf_counter()
                if name == "Ridge":
                    model = Ridge(**params).fit(x_train, train[target])
                    pred = model.predict(x_val)
                elif name == "Random Forest":
                    model = RandomForestRegressor(**params, random_state=SEED, n_jobs=2).fit(train[cols], train[target])
                    pred = model.predict(parts["validation"][cols])
                else:
                    model = XGBRegressor(**params, learning_rate=.05, subsample=.85, colsample_bytree=.9,
                        tree_method="hist", objective="reg:squarederror", random_state=SEED, n_jobs=2).fit(train[cols], train[target])
                    pred = model.predict(parts["validation"][cols])
                elapsed = time.perf_counter() - start
                error = metrics(parts["validation"][target], np.maximum(0, pred))["RMSE"]
                candidates.append({"cluster_id": cluster, "target": target, "model": name,
                    "params": json.dumps(params), "validation_RMSE": error, "fit_and_validation_seconds": elapsed})
                if error < best_error:
                    best, best_error, best_params = model, error, params
            start = time.perf_counter()
            predictions[name] = {label: np.maximum(0, best.predict(scaler.transform(part[cols]) if name == "Ridge" else part[cols]))
                                 for label, part in parts.items()}
            timings.append({"cluster_id": cluster, "target": target, "model": name,
                            "selected_params": json.dumps(best_params), "prediction_seconds": time.perf_counter()-start})
            if name == "XGBoost" and hasattr(best, "feature_importances_"):
                importance.extend({"cluster_id": cluster, "target": target, "feature": feature, "gain_normalized": float(value)}
                                  for feature, value in zip(cols, best.feature_importances_))
        for name, pred_parts in predictions.items():
            for label, part in parts.items():
                pred = np.maximum(0, np.asarray(pred_parts[label], dtype=float))
                records.append({"cluster_id": cluster, "target": target, "model": name, "split": label,
                    "train_rows": len(train), "validation_rows": len(parts["validation"]), "test_rows": len(parts["test"]),
                    "evaluation_rows": len(part), "train_end_utc": str(train_end),
                    "validation_end_utc": str(validation_end), "test_end_utc": str(test_end), **metrics(part[target], pred)})
                frames.append(pd.DataFrame({"cluster_id": cluster, "target": target, "model": name,
                    "split": label, "timestamp_utc": part.timestamp_utc.to_numpy(),
                    "y_true": part[target].to_numpy(), "y_pred": pred}))
        print(f"{target} / {cluster}: train={len(train)}, validation={len(parts['validation'])}, test={len(parts['test'])}", flush=True)
    score = pd.DataFrame(records)
    score.attrs["candidates"] = candidates
    score.attrs["timings"] = timings
    score.attrs["importance"] = importance
    return score, pd.concat(frames, ignore_index=True)


## 5. Baseline, Ridge chuẩn hóa train-only, Random Forest và XGBoost

In [5]:
def main(argv=None):
    parser = argparse.ArgumentParser(description="NV5 audited one-step experiment")
    parser.add_argument("--cluster-map", required=True)
    parser.add_argument("--splits-json", required=True)
    parser.add_argument("--output-dir", default=str(OUT))
    args = parser.parse_args(argv)
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    with open(args.splits_json, encoding="utf-8") as f:
        splits = json.load(f)
    _parse_splits(splits)
    df, provenance, source_audit = load_sessions()
    df = attach_clusters(df, args.cluster_map)
    series, series_audit = make_regular_series(df)
    series_audit.update({"source": provenance, **source_audit, "cluster_mapping": df.attrs["cluster_mapping"]})
    (out / "data_audit.json").write_text(json.dumps(series_audit, indent=2), encoding="utf-8")
    series.to_csv(out / "timeseries_15min.csv", index=False)
    scores_all, candidates, timings, importance = [], [], [], []
    for number, target in enumerate(("occupancy_ports", "session_arrivals")):
        scores, predictions = run_target(series, target, splits)
        candidates.extend(scores.attrs["candidates"])
        timings.extend(scores.attrs["timings"])
        importance.extend(scores.attrs["importance"])
        scores.attrs = {}
        scores_all.append(scores)
        predictions.to_csv(out / "baseline_predictions.csv", index=False, mode="w" if number == 0 else "a", header=number == 0)
        del predictions
    comparison = pd.concat(scores_all, ignore_index=True)
    comparison.to_csv(out / "nv5_model_comparison.csv", index=False)
    for label, subset in (("all_clusters", comparison), ("nonconstant_clusters", comparison.loc[~comparison.constant_target])):
        subset.groupby(["target", "split", "model"])[["MAE", "RMSE", "R2"]].mean().to_csv(out / f"summary_{label}.csv")
    pd.DataFrame(candidates).to_csv(out / "validation_candidates.csv", index=False)
    pd.DataFrame(timings).to_csv(out / "model_timings.csv", index=False)
    pd.DataFrame(importance).to_csv(out / "feature_importance.csv", index=False)
    manifest = {"elapsed_seconds": time.perf_counter()-start, "splits": splits, "seed": SEED,
        "forecast": "one step; actual target available through t-1; frozen train models",
        "calendar_timezone": "America/Los_Angeles", "python": platform.python_version(),
        "xgboost_version": sys.modules["xgboost"].__version__,
        "packages": {k: version(k) for k in ("numpy", "pandas", "scikit-learn")},
        "notebook_sha256": hashlib.sha256((ROOT / "notebooks/05_NV5_demand_forecasting.ipynb").read_bytes()).hexdigest(),
        "mapping_sha256": hashlib.sha256(Path(args.cluster_map).read_bytes()).hexdigest(),
        "metric_rows": len(comparison), "limitations": ["No time-varying availability: zero means no observed sessions, not confirmed downtime/free capacity", "Constant-target R2 excluded from nonconstant summary", "No causal or multi-step forecast claim"]}
    (out / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Completed:", out, "seconds:", round(manifest["elapsed_seconds"], 2))
    return comparison


## 6. Kiểm tra dữ liệu đầu vào (chạy được ngay)

Cell này hiện kết quả từ NV1 và lưu audit mà không cần mapping cụm hay mốc chia. Đây là kiểm tra dữ liệu, chưa phải metric dự báo theo cụm.


In [6]:
sessions, data_source, source_audit = load_sessions()
valid_sessions = sessions.loc[sessions.is_valid_session.fillna(False)].copy()
valid_start = _as_datetime(valid_sessions["start_datetime_utc"])
valid_end = _as_datetime(valid_sessions["end_datetime_utc"])
cluster_map_path = ROOT / "data" / "processed" / "station_cluster_map.csv"
splits_path = ROOT / "reports" / "nv5_splits.json"
mapped_sessions = attach_clusters(sessions, cluster_map_path)
expected_ids = mapped_sessions.port_id
id_mismatches = int((sessions.port_id.astype("string").fillna("") != expected_ids.fillna("")).sum())
cluster_map_ready = True
valid_sessions = mapped_sessions.loc[mapped_sessions.is_valid_session.fillna(False)].copy()
audit = {
    "data_source": data_source,
    **source_audit,
    "valid_sessions": int(len(valid_sessions)),
    "valid_zero_duration_sessions": int((valid_end == valid_start).sum()),
    "valid_negative_duration_sessions": int((valid_end < valid_start).sum()),
    "valid_sessions_missing_port_id": int(valid_sessions.port_id.isna().sum()),
    "observed_port_ids": int(expected_ids.nunique()),
    "cluster_mapping": mapped_sessions.attrs["cluster_mapping"],
    "rows_with_port_id_normalized": id_mismatches,
    "cluster_map_ready": bool(cluster_map_ready),
    "split_milestones_file_present": bool(splits_path.exists()),
}
OUT.mkdir(parents=True, exist_ok=True)
pd.Series(audit).to_json(OUT / "nv5_input_audit.json", indent=2, force_ascii=False)
from IPython.display import display
display(pd.DataFrame({"check": list(audit), "result": list(audit.values())}))
print("Input audit saved to:", OUT / "nv5_input_audit.json")


Source: NV1 sessions_features.parquet; audit: {'clean_rows': 259411, 'features_rows': 259411, 'clean_valid_sessions': 259356, 'features_valid_sessions': 259356, 'same_session_keys': True}


,check,result
0,data_source,nv1_parquet
1,clean_rows,259411
2,features_rows,259411
3,clean_valid_sessions,259356
4,features_valid_sessions,259356
5,same_session_keys,True
6,valid_sessions,259356
7,valid_zero_duration_sessions,12
8,valid_negative_duration_sessions,0
9,valid_sessions_missing_port_id,0


Input audit saved to: e:\backup\class\s5\business analyst\doan\project\ev-charging-analytics---IS403\outputs\nv5\nv5_input_audit.json


## 7. Huấn luyện và chấm điểm toàn bộ

Cell dưới thực sự fit mô hình; mỗi lần chạy tạo thư mục mới để không ghi đè kết quả cũ. Đọc run_manifest.json, validation_candidates.csv, model_timings.csv và hai bảng summary. Không dùng lại số liệu mục 4.5 cũ nếu chưa đối chiếu lượt chạy này. Lịch dùng giờ Palo Alto; lưới và ranh giới tập dùng UTC.

Sau khi có thư mục kết quả, chạy `python scripts/evaluate_nv5_robustness.py <run_dir>` từ gốc repo để thêm hai kỳ kiểm tra occupancy và bảng sai số arrivals theo 0, 1, ≥2 phiên.


In [7]:
# Keep previous runs intact. A new folder is created for every full experiment.
run_dir = OUT / pd.Timestamp.now(tz="UTC").strftime("run_%Y%m%dT%H%M%SZ")
main(["--cluster-map", str(ROOT / "data/processed/station_cluster_map.csv"),
      "--splits-json", str(ROOT / "reports/nv5_splits.json"), "--output-dir", str(run_dir)])


Source: NV1 sessions_features.parquet; audit: {'clean_rows': 259411, 'features_rows': 259411, 'clean_valid_sessions': 259356, 'features_valid_sessions': 259356, 'same_session_keys': True}
occupancy_ports / BRYANT: train=259667, validation=17376, test=17664
occupancy_ports / CAMBRIDGE: train=259667, validation=17376, test=17664
occupancy_ports / HAMILTON: train=259667, validation=17376, test=17664
occupancy_ports / HIGH: train=259667, validation=17376, test=17664
occupancy_ports / MPL: train=259667, validation=17376, test=17664
occupancy_ports / RINCONADA LIB: train=259667, validation=17376, test=17664
occupancy_ports / SHERMAN: train=259667, validation=17376, test=17664
occupancy_ports / TED THOMPSON: train=259667, validation=17376, test=17664
occupancy_ports / WEBSTER: train=259667, validation=17376, test=17664
session_arrivals / BRYANT: train=259667, validation=17376, test=17664
session_arrivals / CAMBRIDGE: train=259667, validation=17376, test=17664
session_arrivals / HAMILTON: trai

,cluster_id,target,model,split,train_rows,validation_rows,test_rows,evaluation_rows,train_end_utc,validation_end_utc,test_end_utc,MAE,RMSE,R2,constant_target
0,BRYANT,occupancy_ports,Historical Mean,validation,259667,17376,17664,17376,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,1.446006,2.051275,0.376921,False
1,BRYANT,occupancy_ports,Historical Mean,test,259667,17376,17664,17664,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,1.409269,1.995015,0.388156,False
2,BRYANT,occupancy_ports,Historical Mean,stress_2020,259667,17376,17664,35136,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,1.328592,1.669221,-0.150858,False
3,BRYANT,occupancy_ports,Persistence t-1,validation,259667,17376,17664,17376,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,0.319809,0.668491,0.933826,False
4,BRYANT,occupancy_ports,Persistence t-1,test,259667,17376,17664,17664,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,0.308990,0.656455,0.933754,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,WEBSTER,session_arrivals,Random Forest,test,259667,17376,17664,17664,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,0.262554,0.419059,0.136203,False
374,WEBSTER,session_arrivals,Random Forest,stress_2020,259667,17376,17664,35136,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,0.210090,0.312675,-0.152550,False
375,WEBSTER,session_arrivals,XGBoost,validation,259667,17376,17664,17376,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,0.274874,0.432157,0.149824,False
376,WEBSTER,session_arrivals,XGBoost,test,259667,17376,17664,17664,2019-01-01 00:00:00,2019-07-01 00:00:00,2020-01-01 00:00:00,0.268138,0.420628,0.129725,False
